# Conceptual Overview: Grad-CAM for Pollen Embeddings

**Idea**: Two holographic views of the same pollen sample can look different while still representing the same object. This notebook explores which image regions contribute to the similarity between their learned embeddings.

**Approach**: It loads paired pollen images and a previously trained BYOL representation model. For each pair, the embedding of one view serves as a reference. Grad-CAM then uses the cosine similarity to that reference as its target to highlight regions in the other view that support this similarity. The notebook performs this comparison in both directions, explaining each view using its paired image.

The selected target layer determines which internal feature maps are used to generate the heatmap. Comparing layers allows the notebook to explore how the highlighted regions change from early image features to deeper representations. Layer-by-layer animations show these changes alongside the source and reference images.

**Results**: The notebook produces heatmaps, image overlays, and animations that can be exported as GIFs, such as `gradcam_layers_pair3.gif`. These visualizations provide a qualitative view of what contributes to embedding similarity; they do not measure classification accuracy. 

Based on the [Pixel Attribution for Embeddings tutorial](https://github.com/jacobgil/pytorch-grad-cam/blob/master/tutorials/Pixel%20Attribution%20for%20embeddings.ipynb).

Imports and paths

In [ ]:
import os
import yaml
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML


import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import DataLoader

from pollen_datasets.poleno import PairwiseHolographyImageFolder

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image, preprocess_image

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

from ssl_poleno.model.encoders import BYOLPolenoEmbedding

**Utilities: configuration and image conversion**

In [ ]:
def load(config_file):
    with open(config_file, 'r') as stream:
        try:
            config = yaml.safe_load(stream)
            return config
        except yaml.YAMLError as exc:
            print(exc)
    

def tensor_to_rgb_image(x):
    """
    Convert single grayscale image tensor from (1, H, W)
    to RGB numpy image (H, W, 3).
    """
    image_float = x.permute(1, 2, 0).detach().cpu().numpy()
    image_float = np.repeat(image_float, 3, axis=2)
    image_float = np.clip(image_float, 0, 1)
    return image_float


class GradCAMEmbedding(nn.Module):
    """Enable encoder gradients for Grad-CAM's model(input) calls."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x, no_grad=False)


**Configuration and image transforms**

In [ ]:
config = load("config/base_byol_dual_config_L_effnet.yaml")
dataset_config = config["dataset"]
condition_config = config["conditioning"]


# Transformations
transforms_list = []
transforms_list.append(transforms.ToTensor())

if dataset_config.get("img_interpolation"):
    transforms_list.append(
        transforms.Resize((dataset_config["img_interpolation"], 
                            dataset_config["img_interpolation"]),
                            interpolation = transforms.InterpolationMode.BILINEAR))
    
transforms_list.append(
    transforms.Normalize(
        [0.5] * dataset_config["img_channels"], 
        [0.5] * dataset_config["img_channels"]))

transform = transforms.Compose(transforms_list)


**Paired dataset and data loader**

In [ ]:
# Dataset
dataset = PairwiseHolographyImageFolder(
    root=dataset_config["root"], 
    labels=dataset_config["labels_train"],
    dataset_cfg=dataset_config,
    cond_cfg=condition_config,
    verbose=True,
    transform1=transform,
    transform2=transform,
)

In [ ]:
# Dataloader
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

**BYOL checkpoint and embedding model**

In [ ]:
model = BYOLPolenoEmbedding(
    ckpt_path=r"Z:\simon_luder\BYOL\BYOL_Representations_for_Holographic_Pollen\checkpoints\byol_lit_20260204_152517\best_epoch=15-step=3740-val_loss=0.1027.ckpt", 
    emb_layer="avgpool", 
    out_dim=2048, 
    backbone="resnet50",
)
model.eval()

**Sample batch and reference embeddings**

In [ ]:
(img1, img2), (cond1, cond2), (fn1, fn2) = next((iter(dataloader)))

img1.shape

In [ ]:
# Reference embeddings are fixed targets; only the CAM input needs gradients.
features = model(img2).detach()

**Cosine-similarity target and Grad-CAM overlay**

In [ ]:
class SimilarityToConceptTarget:
    def __init__(self, concept_features):
        self.concept_features = concept_features.detach()

    def __call__(self, model_output):
        cos = torch.nn.CosineSimilarity(dim=0)
        return cos(model_output, self.concept_features)

target_layers = [model.encoder.net.layer4[-1]]
target_layers = [model.encoder.net.conv1]

# Explain each image using its paired reference embedding.
targets = [SimilarityToConceptTarget(feature) for feature in features]

# Batch of grayscale images: shape (4, 1, 224, 224)
input_tensor = img1

# Prepare first image for visualization
image_float = img1[0].permute(1, 2, 0).detach().cpu().numpy()
image_float = np.repeat(image_float, 3, axis=2)

# Optional but often important
image_float = np.clip(image_float, 0, 1)

with GradCAM(model=GradCAMEmbedding(model), target_layers=target_layers) as cam:
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)

# Select first image's CAM
car_grayscale_cam = grayscale_cam[0, :]

# Overlay heatmap on original image
cam_image = show_cam_on_image(image_float, car_grayscale_cam, use_rgb=True)

Image.fromarray(cam_image)

Reference image preview

In [ ]:
plt.imshow(tensor_to_rgb_image(img2[5]))

**Target-layer selection**

The **target layer** defines which internal feature map of the model Grad-CAM uses to compute the heatmap, thereby controlling how detailed or high-level the visual explanation is.
- Input layers: highlight fine surface patterns or boundaries, more detailed, low-level maps
- Later layers: more semantic information but coarser maps, less spatially precise

In [ ]:
import torch
import torch.nn.functional as F

class SimilarityToReferenceTarget:
    def __init__(self, reference_features):
        self.reference_features = reference_features.detach()

    def __call__(self, model_output):
        return F.cosine_similarity(
            model_output,
            self.reference_features,
            dim=0
        )

model.eval()

(img1, img2), (cond1, cond2), (fn1, fn2) = next((iter(dataloader)))


# 1. Get embedding features
with torch.no_grad():
    features1 = model(img1)  # embeddings for img1
    features2 = model(img2)  # embeddings for img2

# target_layers = [model.encoder.net.layer4[-1]]
# target_layers = [model.encoder.net.layer1[0]]
target_layers = [model.encoder.net.conv1]
# target_layers = [model.encoder.net.avgpool]

# Direction A:
# Explain img1[i] using img2[i] as reference
targets_img1_to_img2 = [
    SimilarityToReferenceTarget(features2[i])
    for i in range(img1.shape[0])
]

# Direction B:
# Explain img2[i] using img1[i] as reference
targets_img2_to_img1 = [
    SimilarityToReferenceTarget(features1[i])
    for i in range(img2.shape[0])
]

with GradCAM(model=GradCAMEmbedding(model), target_layers=target_layers) as cam:
    cams_img1_explained_by_img2 = cam(
        input_tensor=img1,
        targets=targets_img1_to_img2
    )

with GradCAM(model=GradCAMEmbedding(model), target_layers=target_layers) as cam:
    cams_img2_explained_by_img1 = cam(
        input_tensor=img2,
        targets=targets_img2_to_img1
    )

**Paired images and heatmaps**

In [ ]:
N = img1.shape[0]

fig, axes = plt.subplots(N, 4, figsize=(14, 3.5 * N))

if N == 1:
    axes = axes[None, :]

for i in range(N):
    image1 = tensor_to_rgb_image(img1[i])
    image2 = tensor_to_rgb_image(img2[i])

    cam1_overlay = show_cam_on_image(
        image1,
        cams_img1_explained_by_img2[i],
        use_rgb=True
    )
    
    cam2_overlay = show_cam_on_image(
        image2,
        cams_img2_explained_by_img1[i],
        use_rgb=True
    )

    cam1_overlay = cams_img1_explained_by_img2[i]
    cam2_overlay = cams_img2_explained_by_img1[i]

    axes[i, 0].imshow(image1)
    axes[i, 0].set_title(f"Object {i}: img1")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(cam1_overlay)
    axes[i, 1].set_title("img1 explained by img2")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(image2)
    axes[i, 2].set_title(f"Object {i}: img2")
    axes[i, 2].axis("off")

    axes[i, 3].imshow(cam2_overlay)
    axes[i, 3].set_title("img2 explained by img1")
    axes[i, 3].axis("off")

plt.tight_layout()
plt.show()

**Layer-by-layer Grad-CAM computation**

In [ ]:
def compute_layerwise_cams_for_pair(
    model,
    img1,
    img2,
    pair_idx,
    layer_list,
    layer_names=None,
    direction="img1_to_img2"
):
    """
    Compute Grad-CAM overlays across multiple layers for one paired object.

    direction:
        - "img1_to_img2": explain img1[pair_idx] using img2[pair_idx] as reference
        - "img2_to_img1": explain img2[pair_idx] using img1[pair_idx] as reference
    """
    model.eval()

    x1 = img1[pair_idx:pair_idx+1]
    x2 = img2[pair_idx:pair_idx+1]

    with torch.no_grad():
        f1 = model(x1)
        f2 = model(x2)

    if direction == "img1_to_img2":
        input_tensor = x1
        reference_features = f2[0]
        base_image = tensor_to_rgb_image(x1[0])
        title_prefix = f"Pair {pair_idx}: img1 explained by img2"
    elif direction == "img2_to_img1":
        input_tensor = x2
        reference_features = f1[0]
        base_image = tensor_to_rgb_image(x2[0])
        title_prefix = f"Pair {pair_idx}: img2 explained by img1"
    else:
        raise ValueError("direction must be 'img1_to_img2' or 'img2_to_img1'")

    target = [SimilarityToReferenceTarget(reference_features)]

    overlays = []
    frame_titles = []

    if layer_names is None:
        layer_names = [f"Layer {k}" for k in range(len(layer_list))]

    for layer, layer_name in zip(layer_list, layer_names):
        with GradCAM(model=GradCAMEmbedding(model), target_layers=[layer]) as cam:
            grayscale_cam = cam(
                input_tensor=input_tensor,
                targets=target
            )[0]

        overlay = show_cam_on_image(
            base_image,
            grayscale_cam,
            use_rgb=True
        )

        overlays.append(overlay)
        frame_titles.append(f"{title_prefix}\n{layer_name}")

    return base_image, overlays, frame_titles

Animation helper

In [ ]:
def animate_layerwise_cams(
    model,
    img1,
    img2,
    pair_idx,
    layer_list,
    layer_names=None,
    direction="img1_to_img2",
    interval=1000
):
    """
    Create a notebook animation of Grad-CAM through multiple layers for one pair.
    """
    base_image, overlays, frame_titles = compute_layerwise_cams_for_pair(
        model=model,
        img1=img1,
        img2=img2,
        pair_idx=pair_idx,
        layer_list=layer_list,
        layer_names=layer_names,
        direction=direction
    )

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    axes[0].imshow(base_image)
    axes[0].set_title("Original image")
    axes[0].axis("off")

    im = axes[1].imshow(overlays[0])
    axes[1].set_title(frame_titles[0])
    axes[1].axis("off")

    def update(frame):
        im.set_data(overlays[frame])
        axes[1].set_title(frame_titles[frame])
        return [im]

    anim = FuncAnimation(
        fig,
        update,
        frames=len(overlays),
        interval=interval,
        blit=False,
        repeat=True
    )

    plt.close(fig)
    return anim

Target layers and labels

In [ ]:
layer_list = [
    model.encoder.net.conv1,
    model.encoder.net.layer1[0],
    model.encoder.net.layer1[-1],
    model.encoder.net.layer2[-1],
    model.encoder.net.layer3[-1],
    model.encoder.net.layer4[-1],
]

layer_names = [
    "conv1",
    "layer1[0]",
    "layer1[-1]",
    "layer2[-1]",
    "layer3[-1]",
    "layer4[-1]",
]

Layer animation preview

In [ ]:
anim = animate_layerwise_cams(
    model=model,
    img1=img1,
    img2=img2,
    pair_idx=0,
    layer_list=layer_list,
    layer_names=layer_names,
    direction="img1_to_img2",
    interval=1200
)

HTML(anim.to_jshtml())

Animation with source and reference images

In [ ]:
import torch
import torch.nn.functional as F


class SimilarityToReferenceTarget:
    def __init__(self, reference_features):
        self.reference_features = reference_features.detach().flatten()

    def __call__(self, model_output):
        model_output = model_output.flatten()
        return F.cosine_similarity(model_output, self.reference_features, dim=0)


def tensor_to_rgb_image(x):
    """
    Convert tensor image (C,H,W) to RGB numpy image (H,W,3).
    Works for grayscale (1 channel) and RGB (3 channels).
    """
    image = x.detach().cpu().permute(1, 2, 0).numpy()

    if image.shape[2] == 1:
        image = np.repeat(image, 3, axis=2)

    image = np.clip(image, 0, 1)
    return image


def animate_layerwise_cam_with_target(
    model,
    img1,
    img2,
    pair_idx,
    layer_list,
    layer_names=None,
    direction="img1_to_img2",
    interval=1200,
    cmap="jet"
):
    """
    Animate Grad-CAM through multiple layers for one paired object.

    Panels:
    1. source image
    2. target/reference image
    3. raw activation map only
    4. activation overlay on source image

    direction:
        - 'img1_to_img2': explain img1[pair_idx] using img2[pair_idx] as reference
        - 'img2_to_img1': explain img2[pair_idx] using img1[pair_idx] as reference
    """
    model.eval()

    x1 = img1[pair_idx:pair_idx+1]
    x2 = img2[pair_idx:pair_idx+1]

    with torch.no_grad():
        f1 = model(x1)
        f2 = model(x2)

    if direction == "img1_to_img2":
        source_tensor = x1
        target_tensor = x2
        reference_features = f2[0]
        source_title = f"Source: img1[{pair_idx}]"
        target_title = f"Target: img2[{pair_idx}]"
        overlay_title_prefix = "img1 explained by img2"
    elif direction == "img2_to_img1":
        source_tensor = x2
        target_tensor = x1
        reference_features = f1[0]
        source_title = f"Source: img2[{pair_idx}]"
        target_title = f"Target: img1[{pair_idx}]"
        overlay_title_prefix = "img2 explained by img1"
    else:
        raise ValueError("direction must be 'img1_to_img2' or 'img2_to_img1'")

    source_image = tensor_to_rgb_image(source_tensor[0])
    target_image = tensor_to_rgb_image(target_tensor[0])

    target = [SimilarityToReferenceTarget(reference_features)]

    if layer_names is None:
        layer_names = [f"Layer {i}" for i in range(len(layer_list))]

    activation_maps = []
    overlays = []

    for layer in layer_list:
        with GradCAM(model=GradCAMEmbedding(model), target_layers=[layer]) as cam:
            grayscale_cam = cam(
                input_tensor=source_tensor,
                targets=target
            )[0]

        activation_maps.append(grayscale_cam)
        overlays.append(
            show_cam_on_image(source_image, grayscale_cam, use_rgb=True)
        )

    fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))

    # static panels
    axes[0].imshow(source_image)
    axes[0].set_title(source_title)
    axes[0].axis("off")

    axes[1].imshow(target_image)
    axes[1].set_title(target_title)
    axes[1].axis("off")

    # animated panels
    act_im = axes[2].imshow(activation_maps[0], cmap=cmap, vmin=0, vmax=1)
    axes[2].set_title(f"Activations\n{layer_names[0]}")
    axes[2].axis("off")

    overlay_im = axes[3].imshow(overlays[0])
    axes[3].set_title(f"{overlay_title_prefix}\n{layer_names[0]}")
    axes[3].axis("off")

    def update(frame):
        act_im.set_data(activation_maps[frame])
        overlay_im.set_data(overlays[frame])

        axes[2].set_title(f"Activations\n{layer_names[frame]}")
        axes[3].set_title(f"{overlay_title_prefix}\n{layer_names[frame]}")

        return [act_im, overlay_im]

    anim = FuncAnimation(
        fig,
        update,
        frames=len(layer_list),
        interval=interval,
        blit=False,
        repeat=True
    )

    plt.close(fig)
    return anim

Pair selection and animation preview

In [ ]:
anim = animate_layerwise_cam_with_target(
    model=model,
    img1=img1,
    img2=img2,
    pair_idx=3,
    layer_list=layer_list,
    layer_names=layer_names,
    direction="img1_to_img2",
    interval=1400
)

HTML(anim.to_jshtml())

### GIF export

In [ ]:
anim.save("gradcam_layers_pair3.gif", writer="pillow", fps=1)

## Photograph example: pretrained ResNet

In [ ]:
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')
from torchvision.models.segmentation import deeplabv3_resnet50
import torch.functional as F
import requests
import cv2
import torchvision

# A model wrapper that gets a resnet model and returns the features before the fully connected layer.
class ResnetFeatureExtractor(torch.nn.Module):
    def __init__(self, model):
        super(ResnetFeatureExtractor, self).__init__()
        self.model = model
        self.feature_extractor = torch.nn.Sequential(*list(self.model.children())[:-1])
                
    def __call__(self, x):
        return self.feature_extractor(x)[:, :, 0, 0]
        
resnet = torchvision.models.resnet50(pretrained=True)
resnet.eval()
model = ResnetFeatureExtractor(resnet)


def get_image_from_url(url):
    """A function that gets a URL of an image, 
    and returns a numpy image and a preprocessed
    torch tensor ready to pass to the model """

    img = np.array(Image.open(requests.get(url, stream=True).raw))
    img = cv2.resize(img, (512, 512))
    rgb_img_float = np.float32(img) / 255
    input_tensor = preprocess_image(rgb_img_float,
                                   mean=[0.485, 0.456, 0.406],
                                   std=[0.229, 0.224, 0.225])
    return img, rgb_img_float, input_tensor